<div style="background-color:#000047; padding:30px; border-radius:10px; color:white; text-align:center;">
    <img src='Figures/alinco_white_text.png' style="height:100px; margin-bottom:10px;"/>
    <h1>Módulo 3: Modelos de Lenguaje</h1>
    <h2>BERT y RoBERTa: Modelos Pre-entrenados e Inferencia</h2>
</div>


---
## Configuracion del entorno

> **Nota:** la primera ejecucion descarga el modelo BETO (~440 MB). Se cachea en `~/.cache/huggingface`
> y las siguientes ejecuciones son rapidas. Si trabajas sin conexion, descargalo previamente.

In [ ]:
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import os

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs('img', exist_ok=True)
sns.set_theme(style='whitegrid')

import transformers
print('transformers:', transformers.__version__)
print('torch       :', torch.__version__)
print('dispositivo :', device)


<a id="sec1"></a>
## 1. Modelos pre-entrenados y Masked Language Modeling (MLM)

**BERT** (Bidirectional Encoder Representations from Transformers) es una pila de **encoders**
(notebook 1) pre-entrenada sobre enormes corpus con una tarea auto-supervisada: predecir
palabras enmascaradas.

```
Entrada:  El [MASK] negro duerme en el sofa
                ^
                |
          BERT predice: "gato" (0.62), "perro" (0.21), ...
```

A diferencia de Word2Vec (un vector fijo por palabra), BERT produce **embeddings contextuales**:
el vector de "banco" cambia segun la oracion.

| Modelo | Arquitectura | Pre-entrenamiento | Idioma |
|--------|--------------|-------------------|--------|
| BERT (BETO) | Encoder | MLM + NSP | Espanol |
| RoBERTa | Encoder | MLM (mejorado) | Multiples |
| GPT | Decoder | Causal LM | Multiples |

<a id="sec2"></a>
## 2. Tokenizacion por subpalabras

Los Transformers no usan palabras completas sino **subpalabras** (WordPiece en BERT, BPE en
RoBERTa). Esto maneja palabras desconocidas dividiendolas en piezas conocidas:

```
"electroencefalografista"  ->  ['electro', '##ence', '##falo', '##graf', '##ista']
```

El prefijo `##` indica continuacion de la palabra anterior.

In [ ]:
from transformers import AutoTokenizer

MODELO_BERT = 'dccuchile/bert-base-spanish-wwm-cased'   # BETO
tokenizer = AutoTokenizer.from_pretrained(MODELO_BERT)

# Corpus de ejemplo EN ESPANOL (embebido)
oraciones = [
    "El gato negro duerme tranquilo en el sofa.",
    "La inteligencia artificial transforma la sociedad.",
    "Me siento en el banco del parque a leer.",
    "Voy al banco a solicitar un prestamo."
]

for o in oraciones[:2]:
    tokens = tokenizer.tokenize(o)
    print(f'{o}\n  -> {tokens}\n')


In [ ]:
# Visualizacion 1: numero de subpalabras (tokens) por oracion
conteos = [len(tokenizer.tokenize(o)) for o in oraciones]
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(x=[f'Oracion {i+1}' for i in range(len(oraciones))], y=conteos,
            palette='viridis', ax=ax)
for i, c in enumerate(conteos):
    ax.text(i, c + 0.1, str(c), ha='center', fontweight='bold')
ax.set_title('Numero de subpalabras (tokens WordPiece) por oracion')
ax.set_ylabel('# tokens')
plt.tight_layout()
plt.savefig('img/nb2_token_counts.png', dpi=120)
plt.show()
print('Figura guardada en img/nb2_token_counts.png')


<a id="sec3"></a>
## 3. Cargar BERT en espanol (BETO)

Cargamos el modelo completo para obtener representaciones contextuales. La salida
`last_hidden_state` tiene forma `(batch, n_tokens, 768)`: un vector de 768 dimensiones por token.

In [ ]:
from transformers import AutoModel

modelo = AutoModel.from_pretrained(MODELO_BERT).to(device)
modelo.eval()   # modo evaluacion (desactiva dropout)

def obtener_embeddings(texto):
    # Tokenizamos y movemos los tensores al dispositivo
    inputs = tokenizer(texto, return_tensors='pt').to(device)
    with torch.no_grad():                     # sin gradientes: solo inferencia
        salida = modelo(**inputs)
    # last_hidden_state: (1, n_tokens, 768)
    return salida.last_hidden_state.squeeze(0).cpu(), inputs

emb, inputs = obtener_embeddings(oraciones[0])
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'].squeeze(0).cpu())
print('Tokens   :', tokens)
print('Embeddings shape:', tuple(emb.shape), '(n_tokens x 768)')


<a id="sec4"></a>
## 4. Embeddings contextuales y polisemia

La palabra **"banco"** significa cosas distintas en:
- *"Me siento en el **banco** del parque"* (asiento)
- *"Voy al **banco** a solicitar un prestamo"* (institucion financiera)

Extraemos el embedding de "banco" en cada contexto y medimos su **similitud coseno**. Si BERT es
contextual, ambos vectores deben ser **distintos**.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def embedding_de_palabra(texto, palabra):
    emb, inputs = obtener_embeddings(texto)
    toks = tokenizer.convert_ids_to_tokens(inputs['input_ids'].squeeze(0).cpu())
    # Buscamos el indice del primer token que coincide con la palabra
    idx = next(i for i, t in enumerate(toks) if t.lower().startswith(palabra[:4]))
    return emb[idx].numpy()

vec_banco_1 = embedding_de_palabra(oraciones[2], 'banco')   # parque
vec_banco_2 = embedding_de_palabra(oraciones[3], 'banco')   # financiero
vec_gato    = embedding_de_palabra(oraciones[0], 'gato')

sim_banco = cosine_similarity([vec_banco_1], [vec_banco_2])[0, 0]
print(f'Similitud banco(parque) vs banco(financiero): {sim_banco:.3f}')
print('-> Menor a 1.0: BERT distingue los dos significados (contextual).')


In [ ]:
# Visualizacion 2: matriz de similitud entre embeddings contextuales
vectores = np.vstack([vec_banco_1, vec_banco_2, vec_gato])
etiquetas = ['banco\n(parque)', 'banco\n(dinero)', 'gato']
M = cosine_similarity(vectores)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(M, annot=True, fmt='.2f', cmap='rocket_r',
            xticklabels=etiquetas, yticklabels=etiquetas, ax=ax,
            cbar_kws={'label': 'Similitud coseno'})
ax.set_title('Similitud entre embeddings contextuales (BETO)')
plt.tight_layout()
plt.savefig('img/nb2_similitud_contextual.png', dpi=120)
plt.show()
print('Figura guardada en img/nb2_similitud_contextual.png')


<a id="sec5"></a>
## 5. BERT vs RoBERTa

**RoBERTa** (Robustly Optimized BERT Approach) mejora a BERT con:

| Aspecto | BERT | RoBERTa |
|---------|------|---------|
| Tarea NSP (Next Sentence) | Si | **Eliminada** |
| Mascara | Estatica | **Dinamica** |
| Tokenizador | WordPiece | **BPE byte-level** |
| Datos / batch | Menos | **Mas** |
| Resultado | Solido | Generalmente **mejor** |

Visualizamos cómo se proyectan los embeddings de varias palabras con **PCA** (2D).

In [ ]:
from sklearn.decomposition import PCA

# Embeddings (promedio de la oracion) de varias frases para proyectar en 2D
frases = oraciones + [
    "El perro corre feliz por el jardin.",
    "El prestamo bancario tiene intereses altos.",
    "La red neuronal aprende patrones del texto."
]

def embedding_oracion(texto):
    emb, _ = obtener_embeddings(texto)
    return emb.mean(dim=0).numpy()   # mean pooling

X = np.vstack([embedding_oracion(f) for f in frases])
pca = PCA(n_components=2, random_state=SEED)
P = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(P[:, 0], P[:, 1], c=range(len(frases)), cmap='tab10', s=120)
for i, f in enumerate(frases):
    ax.annotate(f[:28] + '...', (P[i, 0], P[i, 1]), fontsize=8,
                xytext=(5, 5), textcoords='offset points')
ax.set_title('Proyeccion PCA de embeddings de oracion (BETO)')
ax.set_xlabel('Componente principal 1')
ax.set_ylabel('Componente principal 2')
plt.tight_layout()
plt.savefig('img/nb2_pca_embeddings.png', dpi=120)
plt.show()
print('Figura guardada en img/nb2_pca_embeddings.png')
